# Module 1 — Exercises (Student Worksheet)
### DeepSets, PFN & EFN · *Sets, Graphs & Symmetry for High-Energy Physics* (TIFR ML School 2026)

This worksheet covers **all five exercises** at the end of
`Module1_DeepSets_PFN_EFN.ipynb`. It is **self-contained**: the *Setup* section below re-imports the
libraries, reloads the JetClass data, and re-defines the DeepSets/PFN/EFN building blocks from the
module, so you can run this notebook on its own.

| # | Exercise | Idea it drives home |
|---|----------|---------------------|
| 1 | **Pooling matters** | `mean` forgets multiplicity; the Deep Sets theorem is stated with `sum` for a reason |
| 2 | **Jet-mass regression** | DeepSets is not just a classifier — the same pool regresses a physical observable |
| 3 | **Multiclass** | one head, four jet types; the confusion matrix is *physics* |
| 4 | **IRC-safety in practice** | the EFN is flat *by construction*; how far does the trained PFN actually drift? |
| 5 | **Attention pooling** | a learned, permutation-invariant pool = a one-line Set-Transformer teaser for Module 3 |

Each section restates the problem, explains the approach and the physics/ML behind it, gives runnable
code, and closes with a short discussion of what the numbers mean.

> **Speed knobs.** `N_PER_GRP`, `MAX_PART` and the per-exercise `epochs` trade accuracy for runtime.
> The defaults train in a few minutes on a laptop (CPU / Apple-MPS). Raise them to sharpen every effect.

> **How to use this worksheet.** Run the **Setup** section as-is (it loads the data and provides the models/helpers from the module). Then complete each exercise where you see `# TODO` and `raise NotImplementedError`. Read the **prompt** above each exercise — it sketches the approach. Check your work against the matching `*_Exercises_Solutions.ipynb`.

## Setup — imports, data, and the Module-1 building blocks

Nothing new here: this is the machinery from Module 1, gathered in one place. We load a **balanced,
four-group** JetClass sample (QCD, Higgs, W/Z, top) *once* — that single sample feeds the multiclass
exercise (§3) and the mass-regression exercise (§2), and we carve the **top-vs-QCD** binary subset out
of it for the IRC (§4) and attention-pooling (§5) exercises.

In [ ]:
import os, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

torch.manual_seed(0); np.random.seed(0)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

In [ ]:
# --- Module-1 building blocks (verbatim) -------------------------------------------------
def make_mlp(in_dim, hidden, out_dim, act=nn.ReLU):
    dims = [in_dim, *hidden]; layers = []
    for a, b in zip(dims[:-1], dims[1:]):
        layers += [nn.Linear(a, b), act()]
    layers += [nn.Linear(dims[-1], out_dim)]
    return nn.Sequential(*layers)

def masked_sum(x, mask):    # x:(B,N,C) mask:(B,N)
    return (x * mask.unsqueeze(-1)).sum(dim=1)

def masked_mean(x, mask):
    return masked_sum(x, mask) / mask.sum(dim=1, keepdim=True).clamp(min=1.0)

class DeepSetsClassifier(nn.Module):
    """Canonical Deep Sets / PFN:  out = F( pool_i Phi(x_i) ).  (n_classes=1 => regression head.)"""
    def __init__(self, in_features, phi_hidden=(64, 128, 128), latent=128,
                 f_hidden=(128, 64), n_classes=2, pool="sum"):
        super().__init__()
        self.pool = pool
        self.phi  = make_mlp(in_features, phi_hidden, latent)
        self.f    = make_mlp(latent, f_hidden, n_classes)
    def forward(self, x, mask):
        h = self.phi(x)
        z = masked_sum(h, mask) if self.pool == "sum" else masked_mean(h, mask)
        return self.f(z)

class EFN(nn.Module):
    """Energy Flow Network (IRC-safe):  F( sum_i z_i * Phi(angles_i) )."""
    def __init__(self, n_angle=2, phi_hidden=(64, 128, 128), latent=128,
                 f_hidden=(128, 64), n_classes=2):
        super().__init__()
        self.phi = make_mlp(n_angle, phi_hidden, latent)
        self.f   = make_mlp(latent, f_hidden, n_classes)
    def forward(self, angles, z, mask):
        h = self.phi(angles)
        w = (z * mask).unsqueeze(-1)
        return self.f((w * h).sum(dim=1))

def background_rejection(y, p, target_eff=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    return 1.0 / max(np.interp(target_eff, tpr, fpr), 1e-12)

In [ ]:
# --- Locate + load the JetClass example file --------------------------------------------
import uproot, awkward as ak

CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None:
    raise FileNotFoundError("JetClass_example_100k.root not found; searched:\n  " + "\n  ".join(CANDIDATE_PATHS))
print("Using:", root_path)

MAX_PART   = 128     # pad / truncate every jet to this many particles
N_PER_GRP  = 4000    # jets per GROUP (QCD, H, V, top); raise for accuracy, lower for speed

tree = uproot.open(root_path)["tree"]
LABELS10 = ["label_QCD","label_Hbb","label_Hcc","label_Hgg","label_H4q","label_Hqql",
            "label_Zqq","label_Wqq","label_Tbqq","label_Tbl"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","part_deta","part_dphi",
                  "jet_sdmass","jet_pt"] + LABELS10)

In [ ]:
# --- Build a balanced 4-group sample: 0=QCD, 1=Higgs, 2=W/Z (V), 3=top -------------------
onehot = np.stack([ak.to_numpy(br[k]).astype(bool) for k in LABELS10], axis=1)  # (Njet,10)
GROUPS = {
    0: ["label_QCD"],
    1: ["label_Hbb","label_Hcc","label_Hgg","label_H4q","label_Hqql"],
    2: ["label_Zqq","label_Wqq"],
    3: ["label_Tbqq","label_Tbl"],
}
GROUP_NAMES = ["QCD", "H", "W/Z", "top"]
col = {k: i for i, k in enumerate(LABELS10)}

rng = np.random.default_rng(0)
sel, y4 = [], []
for g, keys in GROUPS.items():
    in_g = np.any(onehot[:, [col[k] for k in keys]], axis=1)
    idx = np.where(in_g)[0]
    idx = rng.permutation(idx)[:N_PER_GRP]
    sel.append(idx); y4.append(np.full(len(idx), g))
    print(f"group {g} ({GROUP_NAMES[g]:>4}): {len(idx)} jets")
sel = np.concatenate(sel); y4 = np.concatenate(y4).astype(np.int64)

# per-particle kinematics for the selected jets (still jagged)
px, py, pz, e = br["part_px"][sel], br["part_py"][sel], br["part_pz"][sel], br["part_energy"][sel]
deta, dphi    = br["part_deta"][sel], br["part_dphi"][sel]
sdmass = ak.to_numpy(br["jet_sdmass"][sel]).astype(np.float32)   # target for the regression exercise

In [ ]:
# --- The 7 Module-1 per-particle features + EFN inputs (angles, energy fraction) ---------
pt = np.sqrt(px**2 + py**2); dR = np.sqrt(deta**2 + dphi**2)
sumpt = ak.sum(pt, axis=1); sume = ak.sum(e, axis=1)
FEATURE_NAMES = ["deta","dphi","log_pt","log_e","log_pt_rel","log_e_rel","dR"]
feat_list = [deta, dphi, np.log(pt+1e-8), np.log(e+1e-8),
             np.log(pt/sumpt+1e-8), np.log(e/sume+1e-8), dR]

def pad_feature(f):
    return ak.to_numpy(ak.fill_none(ak.pad_none(f, MAX_PART, clip=True), 0.0))

X      = np.stack([pad_feature(f) for f in feat_list], axis=-1).astype(np.float32)  # (Njet,MAX_PART,7)
nparts = np.minimum(ak.to_numpy(ak.num(pt, axis=1)), MAX_PART)
mask   = (np.arange(MAX_PART)[None, :] < nparts[:, None]).astype(np.float32)
angles = np.stack([pad_feature(deta), pad_feature(dphi)], axis=-1).astype(np.float32)
zfrac  = pad_feature(pt/sumpt).astype(np.float32)
print("X:", X.shape, " mask:", mask.shape, " angles:", angles.shape, " sdmass:", sdmass.shape)

In [ ]:
# --- Tensors, a shuffled 70/15/15 split, and TRAIN-statistics standardization ------------
Xt      = torch.from_numpy(X)
Mt      = torch.from_numpy(mask)
Y4      = torch.from_numpy(y4)
AnglesT = torch.from_numpy(angles)
Zt      = torch.from_numpy(zfrac)
SDt     = torch.from_numpy(sdmass)

g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(Y4), generator=g)
Xt, Mt, Y4, AnglesT, Zt, SDt = Xt[perm], Mt[perm], Y4[perm], AnglesT[perm], Zt[perm], SDt[perm]

n = len(Y4); n_tr, n_va = int(0.70*n), int(0.15*n)
sl = {"train": slice(0, n_tr), "val": slice(n_tr, n_tr+n_va), "test": slice(n_tr+n_va, n)}

flat = Xt[sl["train"]][Mt[sl["train"]].bool()]
feat_mean, feat_std = flat.mean(0), flat.std(0).clamp(min=1e-6)
Xt = (Xt - feat_mean) / feat_std     # padded slots become non-zero but masked pooling drops them
print("standardized train means ~0:", Xt[sl['train']][Mt[sl['train']].bool()].mean(0).numpy().round(2))

# Binary top-vs-QCD view (labels 0=QCD, 3=top -> 0/1) used by §4 and §5
binmask = (Y4 == 0) | (Y4 == 3)
Yb = (Y4 == 3).long()                # 1 = top, 0 = QCD
print("binary top-vs-QCD jets:", int(binmask.sum().item()))

In [ ]:
# --- Tiny dataset/loader + generic train loop (shared by all exercises) ------------------
class TensorSet(Dataset):
    def __init__(self, tensors): self.tensors = tensors
    def __len__(self): return len(self.tensors[0])
    def __getitem__(self, i): return tuple(t[i] for t in self.tensors)

def loaders_from(tensors_by_split, bs=256):
    return {k: DataLoader(TensorSet(t), batch_size=bs, shuffle=(k == "train"))
            for k, t in tensors_by_split.items()}

def train_classifier(model, loaders, forward, epochs=8, lr=1e-3, n_out=2, quiet=False):
    """forward(model, batch)->(logits, y). Reports val AUC (binary) or accuracy (multiclass)."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        for batch in loaders["train"]:
            batch = [b.to(device) for b in batch]
            logits, y = forward(model, batch)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
        if not quiet:
            m = eval_classifier(model, loaders["val"], forward, n_out)
            extra = f"AUC {m['auc']:.4f}" if n_out == 2 else f"acc {m['acc']:.4f}"
            print(f"  epoch {ep+1:2d}: val loss {m['loss']:.3f}  {extra}")
    return model

@torch.no_grad()
def eval_classifier(model, loader, forward, n_out=2):
    model.eval(); ys, ps, loss_sum, ntot = [], [], 0.0, 0
    for batch in loader:
        batch = [b.to(device) for b in batch]
        logits, y = forward(model, batch)
        loss_sum += F.cross_entropy(logits, y, reduction="sum").item(); ntot += y.size(0)
        ys.append(y.cpu()); ps.append(F.softmax(logits, -1).cpu())
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy()
    out = {"loss": loss_sum/ntot, "y": y, "prob": p, "pred": p.argmax(1),
           "acc": (p.argmax(1) == y).mean()}
    if n_out == 2:
        out["auc"] = roc_auc_score(y, p[:, 1]); out["p1"] = p[:, 1]
    return out

print("Setup complete.")

## Exercise 1 — Pooling matters

> *Re-train the warm-up classifier with `pool="mean"`. Then construct two point clouds that
> `mean`-pooling cannot tell apart but `sum`-pooling can (hint: duplicate every point). What does this
> say about the Deep Sets theorem's preference for the sum?*

**The concept.** Deep Sets says an invariant function is `ρ(⊕_i φ(x_i))` for a symmetric pool `⊕`.
The theorem is proved with the **sum**. `mean` and `max` are *also* permutation-invariant, so they give
valid Deep Sets — but they are **lossy**:

- **`mean` throws away the multiplicity `N`.** `mean({x}) = mean({x, x, …, x})`. A jet with 10 identical
  prongs and a jet with 100 looks the same after mean-pooling.
- **`max` throws away everything but the coordinate-wise extremes.**

The sum keeps both the *shape* of the set and *how many* points it has (it is `N × mean`). Below we
(a) train the image classifier with `sum` and with `mean` and compare accuracy, then (b) build the exact
counterexample the hint asks for: **a cloud and its point-doubled copy**, which are *provably identical*
to a mean-pool network and *provably different* to a sum-pool network.

In [ ]:
# Image point-cloud warm-up (from Module 1 §5), needed only for this exercise.
from torchvision import datasets, transforms

def load_image_dataset(root="../data", train=True):
    tfm = transforms.ToTensor()
    try:
        return datasets.MNIST(root, train=train, download=True, transform=tfm), "MNIST"
    except Exception as ex:
        print("MNIST unavailable (", ex, ") -> FashionMNIST")
        return datasets.FashionMNIST(root, train=train, download=True, transform=tfm), "FashionMNIST"

def image_to_pointcloud(img, threshold=0.0):
    a = img[0]; ys, xs = torch.where(a > threshold); H, W = a.shape
    return torch.stack([xs.float()/W, ys.float()/H, a[ys, xs]], dim=-1)

class PointCloudImages(Dataset):
    def __init__(self, root="../data", train=True, max_points=200):
        self.ds, self.name = load_image_dataset(root, train); self.max_points = max_points
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        img, label = self.ds[idx]; pts = image_to_pointcloud(img)
        if pts.shape[0] > self.max_points:
            pts = pts[torch.argsort(pts[:, 2], descending=True)[:self.max_points]]
        return pts, label

def pad_collate(batch):
    clouds, labels = zip(*batch); N = max(c.shape[0] for c in clouds)
    B, Cd = len(clouds), clouds[0].shape[1]
    x = torch.zeros(B, N, Cd); m = torch.zeros(B, N)
    for i, c in enumerate(clouds):
        x[i, :c.shape[0]], m[i, :c.shape[0]] = c, 1.0
    return x, m, torch.tensor(labels)

train_ds = Subset(PointCloudImages("../data", train=True),  range(20000))
test_ds  = Subset(PointCloudImages("../data", train=False), range(5000))
img_loaders = {"train": DataLoader(train_ds, 128, shuffle=True,  collate_fn=pad_collate),
               "test":  DataLoader(test_ds, 256, shuffle=False, collate_fn=pad_collate)}

In [ ]:
def train_image_model(pool, epochs=5):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 2 — Jet-mass regression

> *Replace the classifier head with a single regression output and train the PFN to predict the jet
> (soft-drop) mass. Use an MSE loss; report the resolution. Which features matter most?*

**The concept.** DeepSets is a general set→vector map; nothing forces the output to be a class. Here we
regress the **soft-drop jet mass** `jet_sdmass` — a real physical observable and, in fact, one of the most
powerful single variables for tagging boosted W/Z/H/top (their masses cluster near the parent-particle
mass, QCD's does not). Two practical points:

1. **Standardize the target.** Masses span 0–200+ GeV; we train on `(m − μ)/σ` and invert for reporting,
   which keeps the MSE well-conditioned.
2. **Report a physicist's resolution**, not just the loss: the median and IQR of the *relative* error
   `(m_pred − m_true)/m_true`, plus the correlation.

For **feature importance** we use *permutation importance*: shuffle one input feature across all real
particles in the test set and measure how much the MSE degrades. A feature the network relies on will,
when scrambled, hurt a lot; an ignored feature will barely move the error.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

In [ ]:
@torch.no_grad()
def predict_mass(model):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
@torch.no_grad()
def test_mse(model, Xin):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 3 — Multiclass tagging

> *Extend from top-vs-QCD to the full set of JetClass labels (QCD, H, W/Z, top). Plot a confusion matrix.
> Which classes are confused, and does that make physical sense?*

**The concept.** The move from binary to multiclass is a **one-line change**: keep the identical PFN and
set the head's `n_classes=4` (softmax cross-entropy handles the rest). What is interesting is the
**structure of the errors**. The four groups are physically ordered by mass/prong-count:
`QCD` (1 prong, low mass) → `W/Z` (2 prongs, ~80–90 GeV) → `H` (often 2 b-prongs, 125 GeV) →
`top` (3 prongs, 173 GeV). We *expect* confusion between **physically adjacent** classes — especially
**W/Z ↔ H** (both are ~2-prong resonances at nearby masses) — and cleaner separation of `top` (its third
prong is a strong handle). The confusion matrix makes this legible.

In [ ]:
def fwd_cls(model, batch):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 4 — How IRC-safe is the PFN in practice?

> *Sweep the collinear/soft deformation strength and plot how far each model's output drifts. The EFN is
> flat by construction — by how much does the trained PFN move?*

**The concept.** An **IRC-safe** observable is unchanged by (i) a **collinear split** — replacing a
particle by two collinear fragments sharing its momentum (fractions `f`, `1−f`) — and (ii) a **soft
emission** — adding a particle with energy fraction `ε → 0`. The **EFN** enforces both *architecturally*:
it pools `Σ_i z_i Φ(n̂_i)` with `z_i = pT_i/Σ pT`, so a collinear split sends `z Φ(n̂) → (fz + (1−f)z)Φ(n̂)`
= unchanged, and a soft particle contributes `z≈0`. The **PFN** feeds every particle's `log pT`, `log E`,
… through `Φ` and has *no such guarantee*.

We first train a PFN and an EFN on top-vs-QCD, then, for many test jets, **sweep the deformation strength**
and measure the mean absolute drift `⟨|p(top)_deformed − p(top)_baseline|⟩`. The EFN curve should sit at
the floor; the PFN curve should rise as we deform harder.

In [ ]:
def split_binary():
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def fwd_pfn(model, batch):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def fwd_efn(model, batch):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
@torch.no_grad()
def pfn_prob(pt, e, deta, dphi):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def efn_prob(pt, e, deta, dphi):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def raw_jet(j):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def collinear_split(pt, e, eta, phi, k, f):     # split particle k into fractions f, 1-f (same angle)
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def add_soft(pt, e, eta, phi, eps):             # add one particle carrying energy fraction eps
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
def sweep(strengths, deform):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 5 — Attention pooling (a teaser for Module 3)

> *Replace `masked_sum` with a learned attention-weighted sum `Σ_i α_i φ(x_i)` where
> `α_i = softmax_i(wᵀ φ(x_i))`. Does it help? You have just built a (very small) Set-Transformer
> pooling layer.*

**The concept.** Sum/mean pooling weights every particle equally. **Attention pooling** lets the network
*learn* how much each particle matters: score each embedded particle with a learnable query vector `w`,
`softmax` over the (masked) set, and take the weighted sum
`z = Σ_i α_i φ(x_i),  α = softmax_i(wᵀ φ(x_i))`. This is still **permutation-invariant** (softmax over a set
+ weighted sum are both symmetric) and still a valid Deep Set — it is exactly **Pooling-by-Multihead-
Attention (PMA)** from the Set Transformer with a single seed, the object Module 3 generalizes. We train it
head-to-head with the sum-pool PFN on top-vs-QCD.

In [ ]:
class AttnPoolPFN(nn.Module):
    """PFN with a learned attention pool:  z = sum_i softmax_i(w . phi(x_i)) * phi(x_i)."""
    def __init__(self, in_features, phi_hidden=(64,128,128), latent=128, f_hidden=(128,64), n_classes=2):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, x, mask):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
def fwd_attn(model, batch):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Wrap-up

You've reached the end of the worksheet. If every exercise runs without a `NotImplementedError`, you've implemented them all — compare your results and reasoning with the solutions notebook. The through-line across the course: **every model is constrained message passing** — a choice of relations, a permutation-invariant aggregation, and a baked-in symmetry.